# E8 y E13: regresión logística L2

SGD con L2 sobre las 52 features estandarizadas (escalador ajustado en MPUD). Preentrenamiento con tasa 0.01; ajuste fino con tasa 0.001 sobre los mismos pesos.

Criterios fijados antes en `README.md` (fase F): AUC = mediana del AUC por persona, dejando una persona fuera (18 pliegues); el modelo tiene que superar a la persistencia; el preentrenamiento con MPUD se queda solo si suma 0.01 de AUC o más (E13).

In [1]:
import os, json
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
import numpy as np
import pandas as pd
from experiments.common import pseudonym
from experiments.modeling import load, feature_cols, persistence_score, BLOCKS
from experiments import tabular as T
pd.set_option("display.width", 160)

## Datos

In [2]:
real, mpud = load("real"), load("mpud")
cols = feature_cols(real)
print(f"real: {len(real)} examples, {real.user_id.nunique()} people, y = {real.y.mean():.1%}")
print(f"mpud: {len(mpud)} examples, {mpud.user_id.nunique()} people, y = {mpud.y.mean():.1%}")
print(f"{len(cols)} features")

real: 16281 examples, 18 people, y = 43.9%
mpud: 193980 examples, 196 people, y = 40.3%
52 features


## Baseline de persistencia

El score es la etiqueta de la última hora (F₆₀ o S₆₀ sobre el perfil).

In [3]:
results = {"persistence": T.report(real, persistence_score(real).to_numpy())}
T.summary(results)

,AUC median,CI95,AUC min,AUC max,AUC pooled,PR-AUC,Brier,Onset AUC
persistence,0.567032,0.532-0.577,0.47333,0.678226,0.589602,0.495204,0.395983,0.5


## Preentrenamiento en MPUD

Parada temprana con 20 personas de MPUD apartadas.

In [4]:
pre = T.pretrain_on_mpud(T.LogReg, mpud, cols)

## Dejando una persona fuera

Dos brazos: solo datos reales, y MPUD + ajuste fino. En cada pliegue, 3 personas de entrenamiento son la validación de la parada temprana.

In [5]:
scores_real = T.run_loso(T.LogReg, real, cols, pre=None)
scores_mpud = T.run_loso(T.LogReg, real, cols, pre=pre)
results["lr_real_only"] = T.report(real, scores_real)
results["lr_mpud_finetuned"] = T.report(real, scores_mpud)
T.summary(results)

,AUC median,CI95,AUC min,AUC max,AUC pooled,PR-AUC,Brier,Onset AUC
persistence,0.567032,0.532-0.577,0.47333,0.678226,0.589602,0.495204,0.395983,0.5
lr_real_only,0.530215,0.502-0.605,0.336766,0.746821,0.53641,0.494945,0.26205,0.525783
lr_mpud_finetuned,0.58137,0.542-0.593,0.367302,0.73851,0.595384,0.541029,0.24313,0.552715


## Partición temporal (secundaria)

Entrenar con el primer 60 % de los días de ejemplo de cada persona y probar con el resto.

In [6]:
temporal = {}
test, s = T.run_temporal(T.LogReg, real, cols, pre=None)
temporal["lr_real_only"] = T.report(test, s)
test, s = T.run_temporal(T.LogReg, real, cols, pre=pre)
temporal["lr_mpud_finetuned"] = T.report(test, s)
temporal["persistence"] = T.report(test, persistence_score(test).to_numpy())
T.summary(temporal)

,AUC median,CI95,AUC min,AUC max,AUC pooled,PR-AUC,Brier,Onset AUC
lr_real_only,0.545215,0.513-0.600,0.242857,0.730638,0.592946,0.529463,0.247797,0.564655
lr_mpud_finetuned,0.535784,0.500-0.599,0.321006,0.72113,0.603478,0.54119,0.232197,0.552941
persistence,0.551023,0.500-0.583,0.451587,0.634425,0.571412,0.450331,0.401779,0.5


## Decisión E13 y comparación con la persistencia

In [7]:
auc = {k: r["auc_median"] for k, r in results.items()}
gain = auc["lr_mpud_finetuned"] - auc["lr_real_only"]
best_arm = "lr_mpud_finetuned" if gain >= 0.01 else "lr_real_only"
print(f"E13: MPUD adds {gain:+.3f} AUC -> keep MPUD: {gain >= 0.01}; arm used: {best_arm}")
print(f"beats persistence: {auc[best_arm] > auc['persistence']} ({auc[best_arm]:.3f} vs {auc['persistence']:.3f})")
pd.Series(results[best_arm]["auc_per_person"]).rename(index=pseudonym).sort_values().round(3).to_frame("AUC por persona")

E13: MPUD adds +0.051 AUC -> keep MPUD: True; arm used: lr_mpud_finetuned
beats persistence: True (0.581 vs 0.567)


,AUC por persona
p5029222a,0.367
p7c4409f4,0.510
p8fbf383d,0.513
pf63a9323,0.536
p41a7ccfe,0.542
p082530f5,0.561
pc8511158,0.562
pf518bab4,0.581
p83cae583,0.581
pc19f229f,0.583


## Ablación por bloque (E8)

Con el brazo elegido, se quita un bloque de features a la vez.

In [8]:
ablation = {}
for block, pick in BLOCKS.items():
    keep = [c for c in cols if c not in pick(cols)]
    pre_b = T.pretrain_on_mpud(T.LogReg, mpud, keep) if best_arm.endswith("mpud_finetuned") else None
    r = T.report(real, T.run_loso(T.LogReg, real, keep, pre=pre_b))
    ablation[block] = {"features_removed": len(cols) - len(keep), "auc_median": r["auc_median"],
                       "drop": auc[best_arm] - r["auc_median"]}
pd.DataFrame(ablation).T.round(3).sort_values("drop", ascending=False)

,features_removed,auc_median,drop
profile,12.0,0.566,0.016
w15,11.0,0.568,0.014
current,3.0,0.582,-0.000
w60,12.0,0.585,-0.004
time_context,4.0,0.585,-0.004
day,10.0,0.590,-0.009


## Guardar

In [9]:
out = {"loso": results, "temporal": temporal, "e13_gain": gain, "best_arm": best_arm, "ablation": ablation}
Path("data/reports").mkdir(parents=True, exist_ok=True)
Path("data/reports/e8_lr.json").write_text(json.dumps(out, indent=2, default=float))
print("-> data/reports/e8_lr.json")

-> data/reports/e8_lr.json
